In [ ]:
import torch
import torchvision
import torchaudio

print(torch.__version__)
print(torchvision.__version__)
print(torchaudio.__version__)
print(torch.cuda.is_available())

In [ ]:
import xformers
print(xformers.__version__)

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
from pathlib import Path
from tqdm import tqdm
import random
import shutil

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# .env 파일 로드
load_dotenv()

# 환경변수에서 토큰 가져오기
huggingface_token = os.getenv("HF_TOKEN")

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    "Lykon/dreamshaper-8",
    torch_dtype=torch.float16,
    use_auth_token=huggingface_token,
    use_safetensors=True
).to("cuda")

pipe.enable_xformers_memory_efficient_attention()

pipe.enable_vae_slicing()

pipe.enable_vae_tiling()

In [ ]:
output_dir = Path('mask_images')
image_size = (640, 640)
batch_size = 16
images_per_class = 500
num_inference_steps = 25

In [ ]:
genders = [
    "asian man", "asian woman", "white man", "white woman",
    "black man", "black woman", "latino man", "latino woman",
    "middle eastern man", "middle eastern woman"
]

In [ ]:
poses = [
    "front view",
    "side view",
    "three-quater view",
    "looking up",
    "looking down",
    "head tilted left",
    "head tilted right"
]

In [ ]:
backgrounds = [
    "indoor", "outdoor", "office", "street", "cafe",
    "school", "subway", "park", "shopping mall",
    "restaurant", "library"
]

In [ ]:
clothes = [
    "in casual clothes", "wearing a hoodie", "in a business suit",
    "in a school uniform", "wearing sportwear", "wearing traditional clothes"
]

In [ ]:
mask_colors = ["white", "beige", "blue", "black", "gray", "pink"]

In [ ]:
expressions = [
    "smiling",
    "neutral expression",
    "frowning",
    "surprised expression",
    "serios expression",
    "happy expression"
]

In [ ]:
shot_types = [
    "close-up face",
    "portrait, head and shoulders",
    "upper body portrait",
    "full body portrait"
]

In [ ]:
class_prompt_templates = {
    "mask_on": (
        "a {shot} of a {gender} wearing a {mask_color} face mask covering mouth and nose properly, "
        "{expression}, {pose}, {background}, {clothes}, realistic, high-quality"
    ),

    "no_mask": (
        "a {shot} of a {gender} wearing a {mask_color} face mask covering mouth and nose properly, "
        "{expression}, {pose}, {background}, {clothes}, realistic, high-quality"
    )
}

In [ ]:
negative_prompts = {
    "mask_on": "blurry, deformed face, low quality, unnatural skin",
    "no_mask": "blurry, deformed face, low quality, unnatural skin"
}

In [ ]:
for class_name, template in class_prompt_templates.items():

    class_folder = output_dir / class_name

    class_folder.mkdir(parents=True, exist_ok=True)

    for i in tqdm(range(0, images_per_class, batch_size), desc=class_name):

        prompts, negs = [], [negative_prompts[class_name]] * batch_size

        for _ in range(batch_size):

            prompt = template.format(
                shot=random.choice(shot_types),
                gender=random.choice(genders),
                pose=random.choice(poses),
                background=random.choice(backgrounds),
                clothes=random.choice(clothes),
                mask_color=random.choice(mask_colors),
                expression=random.choice(expressions)
            )

            prompts.append(prompt)

        with torch.inference_mode(), torch.autocast("cuda"):

            images = pipe(prompts, negative_prompt=negs,
                          height=image_size[1],
                          width=image_size[0],
                          num_inference_steps=num_inference_steps).images

        for idx, img in enumerate(images):

            img.save(class_folder / f"{class_name}_{i+idx:05}.png")